# RLHFlow PRM Wrapper — Score Inspection

Inspect the `RLHFlowPRM` wrapper class (`rlhflow_prm.py`). Scores the
same flamingo toy example used in `examine_prm_scores_llama_v1` via
both the single-step and batched API paths and confirms they agree.

In [ ]:
import gc

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from notebook_utils import gpu_mem_used_gb
from rlhflow_prm import RLHFlowPRM

In [2]:
# Model paths
base_dir = "/groups/chichengz/tnn/datasets/"
rlhflow_prm_dir = base_dir + "Llama3.1-8B-PRM-Deepseek-Data"


In [3]:
# Llama 8B PRM fits comfortably on a 32GB V100 in fp16.
prm = RLHFlowPRM(rlhflow_prm_dir)

print(f"plus/minus tokens: {prm.candidate_token_ids}")
print(f"marker token id  : {prm.marker_token_id}")
print(f"dtype            : {next(prm.model.parameters()).dtype}")
print(f"GPU memory used  : {gpu_mem_used_gb():.2f} GB")


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.24it/s]

plus/minus tokens: [10, 12]
marker token id  : 17165
dtype            : torch.bfloat16
GPU memory used  : 15.39 GB


In [ ]:
# Same toy example used in the Qwen PRM smoke test.
problem = (
    "Sue lives in a fun neighborhood.  One weekend, the "
    "neighbors decided to play a prank on Sue.  On Friday "
    "morning, the neighbors placed 18 pink plastic flamingos "
    "out on Sue's front yard.  On Saturday morning, the "
    "neighbors took back one third of the flamingos, painted "
    "them white, and put these newly painted white flamingos "
    "back out on Sue's front yard.  Then, on Sunday morning, "
    "they added another 18 pink plastic flamingos to the "
    "collection. At noon on Sunday, how many more pink "
    "plastic flamingos were out than white plastic flamingos?"
)

reasoning_steps = [
    "To find out how many more pink plastic flamingos were "
    "out than white plastic flamingos at noon on Sunday, we "
    "can break down the problem into steps. First, on Friday, "
    "the neighbors start with 18 pink plastic flamingos.",

    "On Saturday, they take back one third of the flamingos. "
    "Since there were 18 flamingos, (1/3 \\times 18 = 6) "
    "flamingos are taken back. So, they have (18 - 6 = 12) "
    "flamingos left in their possession. Then, they paint "
    "these 6 flamingos white and put them back out on Sue's "
    "front yard. Now, Sue has the original 12 pink flamingos "
    "plus the 6 new white ones. Thus, by the end of Saturday, "
    "Sue has (12 + 6 = 18) pink flamingos and 6 white "
    "flamingos.",

    "On Sunday, the neighbors add another 18 pink plastic "
    "flamingos to Sue's front yard. By the end of Sunday "
    "morning, Sue has (18 + 18 = 36) pink flamingos and "
    "still 6 white flamingos.",

    "To find the difference, subtract the number of white "
    "flamingos from the number of pink flamingos: "
    "(36 - 6 = 30). Therefore, at noon on Sunday, there were "
    "30 more pink plastic flamingos out than white plastic "
    "flamingos. The answer is (\\boxed{30}).",
]

In [ ]:
# Smoke test: one question, one candidate answer whose steps are joined
# by "\n\n". The dual-path API handles both single and batched scoring;
# we run both here to sanity-check they agree.
questions = [problem]
outputs = [["\n\n".join(reasoning_steps)]]

scores_single = prm.score(questions, outputs)

print("=== single (per-step calls) ===")
for i, score in enumerate(scores_single[0][0], start=1):
    print(f"step {i}: P(correct) = {score:.4f}")

scores_batched = prm.score(questions, outputs, batch_size=4)

print("\n=== batched (one forward pass per batch) ===")
for i, score in enumerate(scores_batched[0][0], start=1):
    print(f"step {i}: P(correct) = {score:.4f}")

In [ ]:
# Free RLHFlow PRM.
del prm
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")